# 🧠 Mini Proyecto QA: Responder preguntas con Transformer propio

En este notebook adaptarás tu modelo Transformer para que pueda responder preguntas a partir de un contexto. La respuesta se genera como una secuencia de salida (tipo traducción).

In [ ]:
# 📥 Cargar dataset QA y preparar vocabularios
data = []
with open('/mnt/data/mini_qa_dataset.tsv', encoding='utf-8') as f:
    for line in f:
        if line.strip() and not line.startswith('#'):
            contexto, pregunta, respuesta = line.strip().split('\t')
            entrada = pregunta.strip().lower().split() + ['<sep>'] + contexto.strip().lower().split()
            salida = respuesta.strip().lower().split()
            data.append((entrada, salida))

# Crear vocabularios
def build_vocab(sequences):
    vocab = {'<pad>':0, '<sos>':1, '<eos>':2, '<unk>':3, '<sep>':4}
    idx = 5
    for seq in sequences:
        for tok in seq:
            if tok not in vocab:
                vocab[tok] = idx
                idx += 1
    return vocab

src_vocab = build_vocab([s for s,_ in data])
tgt_vocab = build_vocab([t for _,t in data])
inv_tgt_vocab = {v:k for k,v in tgt_vocab.items()}

In [ ]:
# 🔄 Codificación de tensores
import torch
def encode(seq, vocab, max_len):
    ids = [vocab.get(tok, vocab['<unk>']) for tok in seq]
    ids = [vocab['<sos>']] + ids + [vocab['<eos>']]
    if len(ids) < max_len:
        ids += [vocab['<pad>']] * (max_len - len(ids))
    return torch.tensor(ids[:max_len])

max_src_len = max(len(s) for s, _ in data) + 2
max_tgt_len = max(len(t) for _, t in data) + 2

pairs = [(encode(s, src_vocab, max_src_len), encode(t, tgt_vocab, max_tgt_len)) for s,t in data]

from torch.utils.data import DataLoader
dataloader = DataLoader(pairs, batch_size=2, shuffle=True)

## 🏋️ Entrenamiento del modelo (ajustar con tu clase Transformer)

In [ ]:
# Asegúrate de tener definida tu clase `Transformer`
import torch.nn as nn
import torch.optim as optim

model = Transformer(len(src_vocab), len(tgt_vocab), embed_size=64, nhead=2, num_layers=2)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab['<pad>'])
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(30):
    total_loss = 0
    for src, tgt in dataloader:
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:].reshape(-1)
        logits = model(src, tgt_input)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

## 🔍 Inferencia: responder una pregunta nueva

In [ ]:
def responder(pregunta, contexto):
    model.eval()
    tokens = pregunta.lower().split() + ['<sep>'] + contexto.lower().split()
    src_tensor = encode(tokens, src_vocab, max_src_len).unsqueeze(0)
    tgt_indices = [tgt_vocab['<sos>']]
    for _ in range(max_tgt_len):
        tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0)
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor)
        next_token = output[0, -1].argmax().item()
        if next_token == tgt_vocab['<eos>']:
            break
        tgt_indices.append(next_token)
    respuesta = [inv_tgt_vocab[idx] for idx in tgt_indices[1:]]
    return ' '.join(respuesta)

# Ejemplo
print(responder("¿Qué órgano bombea sangre?", "El corazón es el órgano que bombea sangre."))